In [13]:
import torch
import tiktoken
from configs.model import ModelConfig
from configs.training import TrainingConfig
from configs.scheduler import SchedulerConfig
from configs.optimizer import OptimizerConfig
from configs.checkpoint import CheckpointConfig
from models.gpt import GPT
from generation.sample_text import generate, generate_sample_text, text_to_token_ids,token_ids_to_text
from datasets.preprocess import download_the_verdict,train_val_dataloader
from evaluation.losses import cross_entropy_loss,token_accuracy
from trainer.trainer import Trainer


In [14]:
tokenizer = tiktoken.get_encoding("gpt2")

In [24]:
GPT_SMALL = ModelConfig(
    emb_dim=96,
    n_layers=12,
    n_heads=4,
    activation="gelu",
    context_length=24
)


In [25]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GPT(GPT_SMALL).to(device)

### Generate Text

In [26]:
text = "Every effort moves you "
tokenized_text = text_to_token_ids(text, tokenizer)
ids = generate(model, tokenized_text, max_new_tokens=20, context_size=GPT_SMALL.context_length)
response = token_ids_to_text(ids, tokenizer)
print(response)

Every effort moves you  diarrhea


In [27]:
res_text = generate_sample_text(model,device='cpu',tokenizer=tokenizer,text=text,max_new_tokens=20)
print(res_text)

Every effort moves you  diarrhea %inis Culture rotatingbenef misdem playthroughentin Garryiringnegative Alzheimer Middle bewild RigUberoundedchenko Mour


#### Training llm

In [28]:

TRAIN_CONFIG = TrainingConfig(
    epoch=1,
    batch_size=2,
    stride=24,
    context_length=24
)

OPTIMIZER_CONFIG = OptimizerConfig()
SCHEDULER_CONFIG = SchedulerConfig()
CHECKPOINT_CONFIG = CheckpointConfig()

In [29]:
raw_text = download_the_verdict()
train_dataloader,val_dataloader = train_val_dataloader(raw_text, TRAIN_CONFIG)

TrainingConfig(epoch=1, batch_size=2, stride=24, context_length=24, learning_rate=0.0003, weight_decay=0.1, grad_clip=1.0, mixed_precision=False, shuffle=False, num_workers=0, drop_last=True, gradient_accumulation_steps=1, train_data_ratio=0.9)
TrainingConfig(epoch=1, batch_size=2, stride=24, context_length=24, learning_rate=0.0003, weight_decay=0.1, grad_clip=1.0, mixed_precision=False, shuffle=False, num_workers=0, drop_last=True, gradient_accumulation_steps=1, train_data_ratio=0.9)


In [30]:
trainer=Trainer(model,tokenizer,train_dataloader,val_dataloader,device,cross_entropy_loss,token_accuracy,CHECKPOINT_CONFIG,TRAIN_CONFIG,OPTIMIZER_CONFIG,SCHEDULER_CONFIG)

In [31]:
trainer.fit()           ### use arg resume_latest=True or resume_best=True to resume training

100%|██████████| 96/96 [00:42<00:00,  2.27it/s]


Checkpoint saved -> checkpoints\checkpoint_96.pt
Best checkpoint saved -> checkpoints\best_checkpoint.pt
Output text:
 Every effort moves you can
after 1 epoch global step 96 the train loss 9.476239730914434 val loss 8.018880844116211 and train acc| 0.05316840312055623 val acc| 0.060606058686971664 
